# model_selection

> 对应代码：`EconometricsCode/code_in_notes/Chap.2/model_selection.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.2`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.2")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


清空内存、设定 50 个观测并固定随机种子，保证下面的模拟可以复现。

In [2]:
%%stata
clear
set obs 50
set seed 1000


. clear

. set obs 50
Number of observations (_N) was 0, now 50.

. set seed 1000

. 


生成与交叉验证示例相同的数据生成过程：x 取自 [0,3] 均匀分布，y 的真实条件期望是指数函数 exp(x)，加上正态扰动。我们用不同阶数的多项式去拟合它，借此比较各种模型选择准则。

In [3]:
%%stata
gen x=runiform()*3
gen y=exp(x)+rnormal()*2


. gen x=runiform()*3

. gen y=exp(x)+rnormal()*2

. 


初始化解释变量列表 control 为空，并用 scalar K 记录模型中参数的个数（含常数项，所以初始为 1）。

In [4]:
%%stata
local control ""
scalar K=1


. local control ""

. scalar K=1

. 


循环让多项式阶数 i 从 1 增加到 10：每轮生成 x 的 i 次幂并加入解释变量列表，同时把参数个数 K 加 1，然后 quietly 运行 OLS 回归。这样我们得到一系列嵌套模型，阶数越高模型越灵活。

从回归的返回值中取出两个拟合优度指标：R²（e(r2)）度量样本内拟合程度，但它随解释变量个数增加而单调不减，不能单独用来选模型；调整 R²（e(r2_a)）对参数个数做了惩罚，可以在嵌套模型之间比较。

计算两个信息准则：AIC = -2·对数似然 + 2K，BIC = -2·对数似然 + ln(N)·K。两者都在极大化对数似然的基础上对模型复杂度施加惩罚，BIC 的惩罚随样本量增大而更重，因而倾向于选出更精简的模型；AIC 则渐近等价于留一法交叉验证，更偏向预测精度。

再计算小样本修正的 AICc：当样本量 N 相对于参数个数 K 不大时，AIC 的惩罚偏松，AICc 加上修正项 2K(K+1)/(N-K-1)，在小样本下更接近无偏的风险估计。

把当前阶数 i 与五个准则值打印在一行中（_skip 插入空格分隔），方便逐行比较：随着阶数上升，R² 单调上升，而调整 R²、AIC、BIC、AICc 会先降后升，其转折点就是各准则选出的最优阶数。

In [5]:
%%stata
forvalues i=1/10{
    gen x`i'=x^`i'
    local control "`control' x`i'"
    scalar K=K+1
    quietly: reg y `control'
    scalar r2=e(r2)
    scalar r2a=e(r2_a)
    scalar aic=-2*e(ll)+2*K
    scalar bic=-2*e(ll)+log(e(N))*K
	scalar aicc=aic+(2*K*(K+1))/(e(N)-K-1)
    display `i' _skip r2 _skip r2a _skip aic _skip bic _skip aicc
}


. forvalues i=1/10{
  2.     gen x`i'=x^`i'
  3.     local control "`control' x`i'"
  4.     scalar K=K+1
  5.     quietly: reg y `control'
  6.     scalar r2=e(r2)
  7.     scalar r2a=e(r2_a)
  8.     scalar aic=-2*e(ll)+2*K
  9.     scalar bic=-2*e(ll)+log(e(N))*K
 10.         scalar aicc=aic+(2*K*(K+1))/(e(N)-K-1)
 11.     display `i' _skip r2 _skip r2a _skip aic _skip bic _skip aicc
 12. }
1 .82304425 .81935767 242.00644 245.83049 242.26176
2 .92438994 .92117249 201.49093 207.227 202.01267
3 .92863805 .92398401 200.5997 208.24779 201.48859
4 .93004566 .9238275 201.6036 211.16371 202.96723
5 .93022805 .92229942 203.47306 214.9452 205.42655
6 .93025016 .92051763 205.45721 218.84137 208.12388
7 .93166627 .92027731 206.43164 221.72782 209.94383
8 .93188197 .92052897 208.27355 225.48176 212.77355
9 .931924 .91864088 210.24269 229.36292 215.88372
10 .93192098 .91863727 212.24491 233.27717 219.19228

. 
